# Homework 3

I use:
- **Spark** for loading, joining, and cleaning the F1 datasets
- **pandas + scikit-learn** for modeling
- **MLflow** for experiment tracking

Predict a driver's **race points** using F1 race, driver, constructor, and circuit information.

The classroom compute environment raised a `Py4JSecurityException` for `pyspark.ml` components such as `StringIndexer`, so I used Spark for ETL and scikit-learn for the actual model training step.

In [0]:
pip install mlflow

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import mlflow
import mlflow.sklearn

## 1. Load the F1 datasets

### Logic
I load the core F1 tables needed for a driver-race level prediction task:
- `results`
- `races`
- `drivers`
- `constructors`
- `circuits`

These tables provide the target variable, race context, driver attributes, constructor attributes, and circuit context.

In [0]:
base_path = "/Volumes/gr5069/raw/f1_data"

df_results = spark.read.csv(f"{base_path}/results.csv", header=True, inferSchema=True)
df_races = spark.read.csv(f"{base_path}/races.csv", header=True, inferSchema=True)
df_drivers = spark.read.csv(f"{base_path}/drivers.csv", header=True, inferSchema=True)
df_constructors = spark.read.csv(f"{base_path}/constructors.csv", header=True, inferSchema=True)
df_circuits = spark.read.csv(f"{base_path}/circuits.csv", header=True, inferSchema=True)

display(df_results.limit(5))
display(df_races.limit(5))
display(df_drivers.limit(5))
display(df_constructors.limit(5))
display(df_circuits.limit(5))

## 2. Build the modeling table in Spark

### Logic
The observation unit is one **driver in one race**.

I join:
- `results` to `races` on `raceId`
- `results` to `drivers` on `driverId`
- `results` to `constructors` on `constructorId`
- `races` to `circuits` on `circuitId`

I define:
- target = `points`
- numeric features = `grid`, `year`, `round`, `driver_number`
- categorical features = `driver_nationality`, `constructor_nationality`, `circuit_country`

I keep the data prep in Spark so the joins and cleaning stay consistent with the course workflow.

In [0]:
df_model = (
    df_results.alias("res")
    .join(df_races.alias("ra"), on="raceId", how="left")
    .join(df_drivers.alias("dr"), on="driverId", how="left")
    .join(df_constructors.alias("co"), on="constructorId", how="left")
    .join(df_circuits.alias("ci"), on="circuitId", how="left")
    .select(
        F.col("res.raceId"),
        F.col("res.driverId"),
        F.col("res.constructorId"),
        F.col("res.points").cast("double").alias("label"),
        F.col("res.grid").cast("double").alias("grid"),
        F.col("ra.year").cast("double").alias("year"),
        F.col("ra.round").cast("double").alias("round"),
        F.when(F.col("dr.number") == "\\N", None).otherwise(F.col("dr.number")).cast("double").alias("driver_number"),
        F.col("dr.nationality").alias("driver_nationality"),
        F.col("co.nationality").alias("constructor_nationality"),
        F.col("ci.country").alias("circuit_country")
    )
    .filter(F.col("label").isNotNull())
    .filter(F.col("grid").isNotNull())
    .fillna({
        "driver_number": -1.0,
        "driver_nationality": "Unknown",
        "constructor_nationality": "Unknown",
        "circuit_country": "Unknown"
    })
)

display(df_model.limit(10))
print("Modeling rows:", df_model.count())

## 3. Convert the modeling table to pandas

### Logic
Because the environment blocks `pyspark.ml`, I move the final prepared table into pandas and perform the model training with scikit-learn.

In [0]:
pdf = df_model.toPandas()
print(pdf.shape)
pdf.head()

## 4. Encode the categorical variables

### Logic
scikit-learn models require numeric features. I use `pd.get_dummies()` to convert the categorical columns into binary indicator columns.

This replaces the `StringIndexer + OneHotEncoder` step that would normally happen in a Spark ML pipeline.

In [0]:
pdf_encoded = pd.get_dummies(
    pdf,
    columns=["driver_nationality", "constructor_nationality", "circuit_country"],
    drop_first=False
)

print(pdf_encoded.shape)
pdf_encoded.head()

Dummy variables were created using one-hot encoding to convert categorical features such as driver nationality, constructor nationality, and circuit country into binary numeric columns. This allows the model to incorporate categorical information by representing each category as a separate feature, enabling the model to learn category-specific patterns.

## 5. Train/test split

### Logic
I split the data into training and testing sets so that the model is evaluated on unseen observations.

In [0]:
X = pdf_encoded.drop(columns=["label"])
y = pdf_encoded["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 6. Set up MLflow

### Logic
All runs are logged to one MLflow experiment. Each run will record:
- hyperparameters
- metrics
- the trained model
- artifacts

Artifacts logged for each run:
1. predictions CSV
2. feature importance CSV
3. actual-vs-predicted plot

In [0]:
experiment_name = "/Users/rh3330@columbia.edu/hw3_sklearn_points_prediction"
mlflow.set_experiment(experiment_name)
print("Using MLflow experiment:", experiment_name)

## 7. Run at least 10 experiments with different hyperparameters

### Logic
I train 10 Random Forest models with different hyperparameter settings. For each run I log:
- parameters
- RMSE
- MAE
- MSE
- R²
- the trained model
- three artifacts

I use **RMSE** as the main selection metric because this is a regression task.

In [0]:
param_grid = [
    {"n_estimators": 20, "max_depth": 5, "min_samples_split": 2},
    {"n_estimators": 30, "max_depth": 5, "min_samples_split": 2},
    {"n_estimators": 40, "max_depth": 5, "min_samples_split": 2},
    {"n_estimators": 20, "max_depth": 8, "min_samples_split": 2},
    {"n_estimators": 30, "max_depth": 8, "min_samples_split": 2},
    {"n_estimators": 40, "max_depth": 8, "min_samples_split": 2},
    {"n_estimators": 20, "max_depth": 10, "min_samples_split": 4},
    {"n_estimators": 30, "max_depth": 10, "min_samples_split": 4},
    {"n_estimators": 40, "max_depth": 10, "min_samples_split": 4},
    {"n_estimators": 60, "max_depth": 12, "min_samples_split": 4}
]

artifact_root = "/tmp/hw3_sklearn_artifacts"
os.makedirs(artifact_root, exist_ok=True)

run_summaries = []

for i, params in enumerate(param_grid, start=1):
    run_name = f"rf_points_run_{i}"

    with mlflow.start_run(run_name=run_name):
        model = RandomForestRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        mse = mean_squared_error(y_test, preds)
        rmse = mse ** 0.5
        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)

        mlflow.log_params(params)
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)

        mlflow.sklearn.log_model(model, artifact_path="sklearn-model")

        pred_df = pd.DataFrame({
            "actual": y_test.values,
            "prediction": preds
        })
        pred_path = os.path.join(artifact_root, f"{run_name}_predictions.csv")
        pred_df.to_csv(pred_path, index=False)
        mlflow.log_artifact(pred_path, artifact_path="artifacts")

        fi_df = pd.DataFrame({
            "feature": X_train.columns,
            "importance": model.feature_importances_
        }).sort_values("importance", ascending=False)
        fi_path = os.path.join(artifact_root, f"{run_name}_feature_importance.csv")
        fi_df.to_csv(fi_path, index=False)
        mlflow.log_artifact(fi_path, artifact_path="artifacts")

        plt.figure(figsize=(6, 4))
        plt.scatter(y_test, preds)
        plt.xlabel("Actual points")
        plt.ylabel("Predicted points")
        plt.title(f"Actual vs Predicted - {run_name}")
        plot_path = os.path.join(artifact_root, f"{run_name}_actual_vs_predicted.png")
        plt.savefig(plot_path, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(plot_path, artifact_path="artifacts")

        run_summaries.append({
            "run_name": run_name,
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "min_samples_split": params["min_samples_split"],
            "rmse": rmse,
            "mae": mae,
            "mse": mse,
            "r2": r2
        })

summary_df = pd.DataFrame(run_summaries).sort_values(["rmse", "mae"], ascending=[True, True])
summary_df

## 8. Compare all runs and select the best model

### Logic
I compare all 10 runs and select the best run using the **lowest RMSE**. I also inspect MAE and R² to make sure the chosen model is consistently strong.

In [0]:
best_run = summary_df.sort_values(["rmse", "mae"], ascending=[True, True]).iloc[0]
best_run

## 9. Refit the best model and inspect predictions

### Logic
After selecting the best hyperparameters, I refit the best model and inspect a sample of predictions on the test set.

In [0]:
best_params = {
    "n_estimators": int(best_run["n_estimators"]),
    "max_depth": int(best_run["max_depth"]),
    "min_samples_split": int(best_run["min_samples_split"])
}

best_model = RandomForestRegressor(
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    min_samples_split=best_params["min_samples_split"],
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)
best_preds = best_model.predict(X_test)

best_pred_df = pd.DataFrame({
    "actual": y_test.values,
    "prediction": best_preds
})

best_pred_df.head(20)

### Best model run
I selected the best model run by comparing the 10 tracked MLflow runs and choosing the one with the **lowest RMSE** on the test set.

### Why this run is the best
This is a regression problem, so RMSE is an appropriate primary metric because it measures prediction error directly and penalizes larger mistakes more heavily. I also checked MAE and R² to confirm that the selected run was strong across multiple evaluation measures.

### What was logged for each run
For every run, I logged:
- hyperparameters
- model metrics
- the trained model
- a predictions CSV
- a feature importance CSV
- an actual-vs-predicted plot

## 10. Conclusion
The Random Forest model achieves stable but moderate performance (~0.59 R²), and further hyperparameter tuning yields diminishing returns, indicating that model improvement is primarily constrained by feature quality rather than model complexity.

The model successfully integrates race, driver, constructor, and circuit information to generate predictions, but its predictive power is limited due to weak feature representation and target imbalance. While it captures general trends, it fails to accurately model high-performance outcomes and competitive race dynamics.
